# 📦 Gestor de incidencias con proveedores (Compras)
### Proyecto final - Microcredencial UC3M IA Generativa para Todos
**Estudiante:** Jordi March Reshuan

**Problema:** En Compras, las incidencias con proveedores (retrasos, material defectuoso, errores de pedido o de factura, documentos que faltan) llegan como correos y notas sueltas. Se clasifican a ojo, la respuesta depende de quién la lea y el seguimiento se pierde entre el correo y el Excel.

**Usuario:** El comprador o analista de compras que recibe la incidencia y tiene que decidir qué hacer y en qué plazo.

**Qué hace la app:**

```
texto de la incidencia
      │
      ▼
[1] LLM (Qwen3-4B): extrae y clasifica  ──►  JSON {proveedor, pedido, material, tipo, gravedad, ...}
      │
      ▼
[2] Python: valida el JSON (¿etiqueta permitida? ¿el nº de pedido aparece en el texto?)
      │
      ▼
[3] Python: reglas de Compras  ──►  acción recomendada + plazo + ¿escalar?
      │
      ▼
[4] LLM: borrador de respuesta al proveedor  ──►  la persona lo revisa antes de enviarlo
      │
      ▼
[5] Registro y seguimiento (ID, estado, fecha límite, vencidas) en una interfaz Gradio
```

**Idea de diseño:** *el LLM para entender y redactar; el código para el formato y las decisiones.*

> Todos los datos son **sintéticos** (inventados). No hay datos personales ni comerciales reales.

## Cómo ejecutarlo
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.
2. *Entorno de ejecución → Ejecutar todo*. Tarda unos 5 min la primera vez (descarga el modelo, ~8 GB).
3. La app aparece debajo de la última celda (sección 9).

## 0. Instalar dependencias
Versiones de la práctica de Gradio de la S3. El aviso rojo de pip sobre `datasets` no afecta.

In [11]:
!pip install -q -U gradio==6.31.0 transformers==5.22.1 accelerate

## 1. Cargar el modelo y la función `llm()`
**Qwen3-4B-Instruct-2507**: el que mejor clasificó la gravedad de los que caben gratis en una T4 (lo comparé con Qwen2.5-0.5B, 1.5B y 7B). Corre en Colab, así que el texto de las incidencias no sale a ningún servicio externo.

- `apply_chat_template` convierte los mensajes (`system`, `user`, `assistant`) al formato de chat del modelo.
- `do_sample=False`: elige siempre el token más probable → **misma entrada, misma salida** (como *temperature* = 0).

In [12]:
import json, re, os, time
from datetime import date, timedelta
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELO = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float16, device_map="auto")
modelo.eval()
print(f"✅ Modelo cargado · GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def llm(mensajes, max_tokens=350):
    texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    nuevos = salida[0][entrada["input_ids"].shape[1]:]        # solo lo que ha escrito el modelo
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()

print(llm([{"role": "user", "content": "Responde solo 'OK' si me entiendes."}], max_tokens=10))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ Modelo cargado · GPU ocupada: 6.5 GB
OK


## 2. Datos: 20 incidencias sintéticas
Escritas y etiquetadas a mano (`tipo_real`, `gravedad_real`). Cubren los 6 tipos y las 3 gravedades, con textos formales e informales y con y sin nº de pedido. En la app sirven de **ejemplos**; se guardan también como `incidencias_sinteticas.csv`.

In [13]:
DATASET = pd.DataFrame([
 {
  "id": "INC-S01",
  "texto": "URGENTE: Rodamientos Ibéricos nos avisa de que el pedido OC-5102 de 500 rodamientos 6204 no saldrá de su almacén hasta el día 20. La línea 2 se queda sin stock el miércoles y tendríamos que parar.",
  "tipo_real": "retraso_entrega",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S02",
  "texto": "Embalajes del Sur confirma que las 2.000 cajas del pedido OC-5110 llegarán una semana tarde por falta de materia prima. Tenemos stock para unos 12 días.",
  "tipo_real": "retraso_entrega",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S03",
  "texto": "Papelería Central nos dice que el tóner del pedido OC-5120 llegará el lunes en lugar del viernes.",
  "tipo_real": "retraso_entrega",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S04",
  "texto": "En la recepción del pedido OC-5131 de Plásticos Mediterráneo, el 30% de las preformas PET tienen rebabas y no pasan el control de calidad. Las necesitamos para el envasado de mañana.",
  "tipo_real": "calidad_defectuosa",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S05",
  "texto": "Han llegado 3 de los 50 motores del pedido OC-5140 de Motores Vega con la carcasa golpeada. Funcionan, pero el cliente final no los aceptará así. Tenemos margen hasta fin de mes.",
  "tipo_real": "calidad_defectuosa",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S06",
  "texto": "Tornillería Norte: dos cajas del pedido OC-5152 llegaron con el embalaje exterior mojado, pero hemos revisado los tornillos y están bien.",
  "tipo_real": "calidad_defectuosa",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S07",
  "texto": "Aceros Castilla nos ha enviado chapa de 2 mm en vez de 3 mm (pedido OC-5160). No sirve para la prensa y el lunes empieza la fabricación del pedido de nuestro cliente principal.",
  "tipo_real": "error_pedido",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S08",
  "texto": "Del pedido OC-5171 a Suministros Eléctricos Ruiz han llegado 150 bobinas de cable, pero en el albarán pone 200. Con 150 cubrimos estas dos semanas.",
  "tipo_real": "error_pedido",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S09",
  "texto": "Oficina Total nos ha mandado bolígrafos azules en lugar de negros (pedido OC-5180).",
  "tipo_real": "error_pedido",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S10",
  "texto": "La factura FV-8812 de Transportes Ruta Norte incluye un recargo de combustible de 45 € que no está en la tarifa pactada para el pedido OC-5190.",
  "tipo_real": "error_factura",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S11",
  "texto": "Contabilidad avisa: Motores Vega nos ha cobrado dos veces la factura FV-9034 de 12.400 € (pedido OC-5141). Necesitamos el abono antes del cierre de mes.",
  "tipo_real": "error_factura",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S12",
  "texto": "El contenedor del pedido OC-5210 de Hangzhou Valves está retenido en la aduana de Valencia porque falta el certificado de origen. Sin esas válvulas no podemos terminar el proyecto del cliente.",
  "tipo_real": "documentacion",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S13",
  "texto": "Química Levante no ha enviado la ficha de seguridad del nuevo adhesivo del pedido OC-5220. Sin ella el almacén no puede ubicarlo, aunque no lo necesitaremos hasta dentro de 10 días.",
  "tipo_real": "documentacion",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S14",
  "texto": "Falta el albarán firmado del pedido OC-5230 de Embalajes del Sur para cerrar el expediente.",
  "tipo_real": "documentacion",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S15",
  "texto": "Rodamientos Ibéricos nos informa de que cambia su dirección de facturación a partir del 1 de diciembre.",
  "tipo_real": "otro",
  "gravedad_real": "baja"
 },
 {
  "id": "INC-S16",
  "texto": "El comercial de Motores Vega nos comunica que subirán los precios un 8% en todos los pedidos nuevos a partir de enero.",
  "tipo_real": "otro",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S17",
  "texto": "oye, lo de Etiquetas Rápidas sigue sin llegar. Mañana arrancamos la campaña y sin etiquetas no podemos expedir nada",
  "tipo_real": "retraso_entrega",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S18",
  "texto": "Lote L-778 de harina de Molinos Duero (pedido OC-5260): el laboratorio ha detectado humedad fuera de especificación. Hemos bloqueado el lote.",
  "tipo_real": "calidad_defectuosa",
  "gravedad_real": "alta"
 },
 {
  "id": "INC-S19",
  "texto": "Nos ha llegado de Suministros Eléctricos Ruiz material que no hemos pedido: 20 cuadros eléctricos. Están ocupando media zona de recepción.",
  "tipo_real": "error_pedido",
  "gravedad_real": "media"
 },
 {
  "id": "INC-S20",
  "texto": "Buen día, de Agroexport Guayas les comunicamos que por el paro de transporte en la vía a la costa el despacho del pedido OC-5280 (cajas de cartón para banano) se retrasa 4 días. Su embarque sale el viernes.",
  "tipo_real": "retraso_entrega",
  "gravedad_real": "alta"
 }
])

DATASET.to_csv("incidencias_sinteticas.csv", index=False, encoding="utf-8-sig")
print(len(DATASET), "incidencias ·", DATASET["tipo_real"].value_counts().to_dict())
DATASET.head()

20 incidencias · {'retraso_entrega': 5, 'calidad_defectuosa': 4, 'error_pedido': 4, 'documentacion': 3, 'error_factura': 2, 'otro': 2}


,id,texto,tipo_real,gravedad_real
0,INC-S01,URGENTE: Rodamientos Ibéricos nos avisa de que...,retraso_entrega,alta
1,INC-S02,Embalajes del Sur confirma que las 2.000 cajas...,retraso_entrega,media
2,INC-S03,Papelería Central nos dice que el tóner del pe...,retraso_entrega,baja
3,INC-S04,En la recepción del pedido OC-5131 de Plástico...,calidad_defectuosa,alta
4,INC-S05,Han llegado 3 de los 50 motores del pedido OC-...,calidad_defectuosa,media


## 3. Prompt de extracción y clasificación
1. **Reglas concretas en el `system`:** las 7 claves del JSON, `null` si falta un dato y una definición de cada etiqueta.
2. **Few-shot:** le muestro al modelo dos ejemplos resueltos antes de la incidencia real, para que imite el formato JSON y el criterio de gravedad. En este caso, dos incidencias resueltas como turnos `user` → `assistant`. La segunda tiene `num_pedido: null` para enseñar al modelo a no inventarlo.

In [14]:
TIPOS = ["retraso_entrega", "calidad_defectuosa", "error_pedido", "error_factura", "documentacion", "otro"]
GRAVEDADES = ["alta", "media", "baja"]
CAMPOS = ["proveedor", "num_pedido", "material", "tipo", "gravedad", "motivo_gravedad", "resumen"]

PROMPT_SISTEMA = """Eres un analista de compras. Lees incidencias con proveedores (correos, notas del almacén) y devuelves SOLO un objeto JSON, sin explicaciones ni markdown.
Devuelve SIEMPRE estas 7 claves: proveedor, num_pedido, material, tipo, gravedad, motivo_gravedad, resumen.
- Si un dato no aparece en el texto, pon null. No inventes nada, sobre todo números de pedido, fechas o importes.
- num_pedido: cópialo tal cual aparece en el texto (por ejemplo OC-5102).
- tipo: UNA de estas etiquetas:
  retraso_entrega: el material llegará tarde o no ha llegado en la fecha pactada.
  calidad_defectuosa: el material llegó dañado, defectuoso o fuera de especificación.
  error_pedido: llegó otra referencia, otra cantidad o algo que no se pidió.
  error_factura: precio, importe, recargo o factura incorrectos o duplicados.
  documentacion: falta o está mal un documento (certificado, albarán, ficha de seguridad, papeles de aduana).
  otro: cualquier otra cosa (cambio de datos del proveedor, subida de precios anunciada...).
- gravedad: alta, media o baja.
  alta: para o va a parar la producción o una expedición, afecta a un cliente, a la seguridad o a la calidad alimentaria, o el material está bloqueado.
  media: impacto real pero con margen (días de stock, unidades sueltas) o impacto económico relevante.
  baja: administrativo o menor, sin impacto en la operación.
- motivo_gravedad: una frase que cite la pista del texto que justifica la gravedad.
- resumen: una frase de 20 palabras como máximo, en español."""

EJEMPLOS_FEW_SHOT = [
    ("Buenos días, les escribimos de Transportes Ruta Norte. El camión con su pedido OC-4512 (40 palés de cartón ondulado) ha sufrido una avería en Burgos y llegará el jueves en lugar del lunes.",
     {"proveedor": "Transportes Ruta Norte", "num_pedido": "OC-4512", "material": "40 palés de cartón ondulado",
      "tipo": "retraso_entrega", "gravedad": "media",
      "motivo_gravedad": "Llega 3 días tarde y el texto no dice que pare la producción.",
      "resumen": "Avería del camión: el pedido de cartón llega el jueves en vez del lunes."}),
    ("Hola, la factura F-2291 de Química Levante cobra el disolvente a 3,40 €/kg cuando en el pedido pactamos 2,95 €/kg.",
     {"proveedor": "Química Levante", "num_pedido": None, "material": "disolvente",
      "tipo": "error_factura", "gravedad": "baja",
      "motivo_gravedad": "Diferencia de precio en la factura, sin impacto en la operación.",
      "resumen": "La factura F-2291 cobra el disolvente por encima del precio pactado."}),
]

def mensajes_extraccion(texto):
    mensajes = [{"role": "system", "content": PROMPT_SISTEMA}]
    for ejemplo, respuesta in EJEMPLOS_FEW_SHOT:          # few-shot: user -> assistant
        mensajes.append({"role": "user", "content": f"INCIDENCIA:\n{ejemplo}"})
        mensajes.append({"role": "assistant", "content": json.dumps(respuesta, ensure_ascii=False)})
    mensajes.append({"role": "user", "content": f"INCIDENCIA:\n{texto}"})
    return mensajes

## 4. Analizar y validar la respuesta
El LLM devuelve **texto**, no un diccionario. `analizar_incidencia`:
1. busca el `{...}` y lo convierte con `json.loads`; si no puede, avisa en lugar de romper la app;
2. normaliza los vacíos (`""`, `"null"`, `"no consta"` → `None`);
3. si `tipo` o `gravedad` no son etiquetas permitidas, los corrige y **avisa**;
4. **anti-alucinación:** si el nº de pedido no aparece literalmente en el texto, lo borra y avisa.

In [15]:
def extraer_json(respuesta):
    respuesta = re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S)   # por si el modelo "piensa" en voz alta
    m = re.search(r"\{.*\}", respuesta, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def analizar_incidencia(texto):
    """Devuelve (datos, avisos). datos = None si el modelo no devolvió un JSON válido."""
    respuesta = llm(mensajes_extraccion(texto))
    bruto = extraer_json(respuesta)
    if bruto is None:
        return None, [f"El modelo no devolvió un JSON válido. Respuesta: {respuesta[:200]}"]

    avisos = []
    datos = {}
    for c in CAMPOS:
        v = bruto.get(c)
        if isinstance(v, str) and v.strip().lower() in ("", "null", "none", "no consta", "n/a", "desconocido"):
            v = None
        datos[c] = v.strip() if isinstance(v, str) else v

    if datos["tipo"] not in TIPOS:
        avisos.append(f"Tipo no permitido ({datos['tipo']}) → 'otro'. Revisar a mano.")
        datos["tipo"] = "otro"
    if datos["gravedad"] not in GRAVEDADES:
        avisos.append(f"Gravedad no permitida ({datos['gravedad']}) → 'media'. Revisar a mano.")
        datos["gravedad"] = "media"
    if datos["num_pedido"] and str(datos["num_pedido"]) not in texto:
        avisos.append(f"El nº de pedido '{datos['num_pedido']}' no aparece en el texto: posible invención, lo borro.")
        datos["num_pedido"] = None
    if not datos["num_pedido"]:
        avisos.append("Sin nº de pedido: pídeselo al proveedor o búscalo en el ERP.")
    return datos, avisos

## 5. Reglas de Compras: acción y plazo (sin LLM)
La acción y el plazo los decide esta tabla, no el modelo: es **predecible, auditable y fácil de cambiar**. Plazo según gravedad (alta 1 día, media 3, baja 7); la gravedad alta se escala.

In [16]:
PLAYBOOK = {
    "retraso_entrega":    "Pedir nueva fecha confirmada por escrito; si es grave, valorar envío urgente o proveedor alternativo y avisar a Planificación.",
    "calidad_defectuosa": "Bloquear el material (cuarentena), pedir fotos o informe al proveedor y abrir reclamación: reposición o abono.",
    "error_pedido":       "Comparar con la orden de compra y el albarán; pedir reposición de lo que falta o recogida de lo erróneo. No dar entrada en el ERP hasta aclararlo.",
    "error_factura":      "Retener el pago y pedir factura rectificativa o abono con el precio pactado; avisar a Contabilidad.",
    "documentacion":      "Pedir el documento con fecha límite; si bloquea aduana o almacén, escalar al contacto del proveedor y al agente de aduanas.",
    "otro":               "Revisión manual por el comprador responsable.",
}
SLA_DIAS = {"alta": 1, "media": 3, "baja": 7}

def proponer_accion(datos):
    limite = date.today() + timedelta(days=SLA_DIAS[datos["gravedad"]])
    return {"accion": PLAYBOOK[datos["tipo"]],
            "fecha_limite": limite.isoformat(),
            "escalar": datos["gravedad"] == "alta"}

## 6. Borrador de respuesta al proveedor (segunda llamada al LLM)
Recibe los datos ya validados y la acción decidida, para que el correo sea coherente con la decisión. Si falta un dato escribe `[corchetes]`. **El borrador no se envía solo: lo revisa una persona.**

In [17]:
def redactar_respuesta(datos, texto, accion):
    mensajes = [
        {"role": "system", "content": (
            "Eres un comprador de una empresa industrial. Escribe un correo breve (máximo 120 palabras), "
            "cordial y firme al proveedor, en español. Incluye una línea de asunto. "
            "Usa SOLO los datos que te doy; si falta alguno (nº de pedido, fechas, importes), escribe [corchetes] para que lo complete una persona. "
            "No prometas nada que no esté en la acción indicada. Firma como 'Departamento de Compras'.")},
        {"role": "user", "content": (
            f"Incidencia original:\n{texto}\n\n"
            f"Datos extraídos: {json.dumps(datos, ensure_ascii=False)}\n"
            f"Acción decidida: {accion['accion']}\n"
            f"Necesitamos respuesta antes del {accion['fecha_limite']}.")},
    ]
    return llm(mensajes, max_tokens=300)

## 7. Registro y seguimiento
Cada incidencia registrada recibe un ID (`INC-001`…), un estado (`Abierta` → `En curso` → `Cerrada`) y una fecha límite; si pasa la fecha sin cerrarse, cuenta como **vencida**. Se guarda en `registro_incidencias.csv` (dura lo que la sesión de Colab).

In [18]:
COLUMNAS = ["id", "fecha_alta", "proveedor", "num_pedido", "material", "tipo", "gravedad",
            "accion", "fecha_limite", "estado", "resumen"]
ESTADOS = ["Abierta", "En curso", "Cerrada"]
RUTA_REGISTRO = "registro_incidencias.csv"

REGISTRO = pd.read_csv(RUTA_REGISTRO).to_dict("records") if os.path.exists(RUTA_REGISTRO) else []

def guardar_registro():
    pd.DataFrame(REGISTRO, columns=COLUMNAS).to_csv(RUTA_REGISTRO, index=False, encoding="utf-8-sig")

def registrar(datos, accion):
    nuevo_id = f"INC-{len(REGISTRO) + 1:03d}"
    REGISTRO.append({"id": nuevo_id, "fecha_alta": date.today().isoformat(),
                     **{c: datos.get(c) for c in ["proveedor", "num_pedido", "material", "tipo", "gravedad", "resumen"]},
                     "accion": accion["accion"], "fecha_limite": accion["fecha_limite"], "estado": "Abierta"})
    guardar_registro()
    return nuevo_id

def cambiar_estado(id_incidencia, estado):
    for fila in REGISTRO:
        if fila["id"] == id_incidencia:
            fila["estado"] = estado
            guardar_registro()
            return f"✅ {id_incidencia} → {estado}"
    return f"⚠️ No existe {id_incidencia}"

def tabla_registro():
    return pd.DataFrame(REGISTRO, columns=COLUMNAS)

def kpis():
    df = tabla_registro()
    if df.empty:
        return "Todavía no hay incidencias registradas."
    hoy = date.today().isoformat()
    abiertas = df[df["estado"] != "Cerrada"]
    vencidas = abiertas[abiertas["fecha_limite"] < hoy]
    por_tipo = ", ".join(f"{t}: {n}" for t, n in df["tipo"].value_counts().items())
    return (f"**Total:** {len(df)} · **Sin cerrar:** {len(abiertas)} · **Vencidas:** {len(vencidas)} · "
            f"**Gravedad alta sin cerrar:** {(abiertas['gravedad'] == 'alta').sum()}\n\n**Por tipo:** {por_tipo}")

## 8. Prueba de punta a punta (sin interfaz)
INC-S17 es informal y **no tiene nº de pedido**: el modelo debe dejarlo en `null` y el borrador debe usar `[corchetes]`.

In [19]:
texto_prueba = DATASET.loc[DATASET["id"] == "INC-S17", "texto"].iloc[0]
t0 = time.time()
datos, avisos = analizar_incidencia(texto_prueba)
print(f"⏱️ {time.time() - t0:.1f} s\n")
print(json.dumps(datos, ensure_ascii=False, indent=2))
print("Avisos:", avisos)
if datos:
    accion = proponer_accion(datos)
    print("\nAcción:", accion)
    print("\n--- Borrador ---\n" + redactar_respuesta(datos, texto_prueba, accion))

⏱️ 71.9 s

{
  "proveedor": "Etiquetas Rápidas",
  "num_pedido": null,
  "material": "etiquetas",
  "tipo": "retraso_entrega",
  "gravedad": "alta",
  "motivo_gravedad": "Sin etiquetas no se puede expedir nada en la campaña de mañana.",
  "resumen": "Etiquetas Rápidas no llegan y se parará la expedición de la campaña."
}
Avisos: ['Sin nº de pedido: pídeselo al proveedor o búscalo en el ERP.']

Acción: {'accion': 'Pedir nueva fecha confirmada por escrito; si es grave, valorar envío urgente o proveedor alternativo y avisar a Planificación.', 'fecha_limite': '2026-11-07', 'escalar': True}

--- Borrador ---
Asunto: Retraso en entrega de etiquetas – Pedido [num_pedido]  

Estimado equipo de Etiquetas Rápidas,  

Se informa que el envío de etiquetas para el pedido [num_pedido] sigue sin llegar, lo que impide la expedición de la campaña programada para mañana. Dado el impacto en la operación, solicitamos una nueva fecha de entrega confirmada por escrito antes del 2026-11-07. Si el retraso per

## 9. Interfaz Gradio
- **Nueva incidencia:** pegar el texto → *Analizar* → ficha con acción y plazo. *Redactar respuesta* y *Registrar* se activan solo después de analizar.
- El **borrador** se puede editar en la propia caja y copiar con el botón de la esquina.
- **Seguimiento:** indicadores (sin cerrar, vencidas, gravedad alta), vista **Kanban** (una columna por estado; el color marca la gravedad de lo que sigue abierto y las cerradas pasan a gris) o **Tabla**, y cambio de estado.
- `gr.State` guarda el último análisis entre clics: registrar no vuelve a llamar al modelo.

In [20]:
import gradio as gr

ETIQUETAS_EJEMPLOS = {               # (nombre corto que ve el usuario, texto completo que se carga)
    "INC-S01": "Retraso que para la línea · Rodamientos Ibéricos",
    "INC-S11": "Factura cobrada dos veces · Motores Vega",
    "INC-S12": "Contenedor retenido en aduana · Hangzhou Valves",
    "INC-S17": "Correo informal sin nº de pedido · Etiquetas Rápidas",
    "INC-S09": "Bolígrafos del color equivocado · Oficina Total",
}
EJEMPLOS_APP = [(nombre, DATASET.loc[DATASET["id"] == i, "texto"].iloc[0]) for i, nombre in ETIQUETAS_EJEMPLOS.items()]
COLUMNAS_TABLA = ["id", "proveedor", "num_pedido", "tipo", "gravedad", "fecha_limite", "estado", "resumen"]

def ficha_markdown(datos, accion, avisos):
    icono = {"alta": "🔴", "media": "🟠", "baja": "🟢"}[datos["gravedad"]]
    md = (f"### {icono} {datos['tipo'].replace('_', ' ')} · gravedad {datos['gravedad']}\n"
          f"| Proveedor | Pedido | Material |\n|---|---|---|\n"
          f"| {datos['proveedor'] or '—'} | {datos['num_pedido'] or '—'} | {datos['material'] or '—'} |\n\n"
          f"**Resumen:** {datos['resumen'] or '—'}\n\n"
          f"**Por qué esa gravedad:** {datos['motivo_gravedad'] or '—'}\n\n"
          f"**Qué hacer:** {accion['accion']}\n\n"
          f"**Responder antes del:** {accion['fecha_limite']}")
    if accion["escalar"]:
        md += "\n\n⚠️ **Escalar al responsable de Compras.**"
    if avisos:
        md += "\n\n**Revisar:**\n" + "\n".join(f"- {a}" for a in avisos)
    return md

import html
COLOR_GRAVEDAD = {"alta": "#e5484d", "media": "#f5a524", "baja": "#30a46c"}

def kanban_html():
    """Una columna por estado y una tarjeta por incidencia (borde de color = gravedad)."""
    df = tabla_registro()
    hoy = date.today().isoformat()
    texto = lambda x: html.escape(str(x)) if pd.notna(x) and x else "—"
    orden = {"alta": 0, "media": 1, "baja": 2}
    columnas = ""
    for est in ESTADOS:
        filas = df[df["estado"] == est].copy()
        filas["_orden"] = filas["gravedad"].map(orden)
        filas = filas.sort_values(["_orden", "fecha_limite"])          # lo más grave y urgente, arriba
        tarjetas = ""
        for _, f in filas.iterrows():
            cerrada = est == "Cerrada"
            vencida = not cerrada and f["fecha_limite"] < hoy
            # Color = "necesita atención". Cerrada → gris: ya no pide nada, aunque fuera grave.
            color = "#9ca3af" if cerrada else COLOR_GRAVEDAD.get(f["gravedad"], "#888")
            etiqueta = (f'<span style="background:{color};color:white;border-radius:10px;padding:1px 8px;'
                        f'font-size:11px;font-weight:600">{texto(f["gravedad"]).upper()}</span>')
            tarjetas += (
                f'<div style="border-left:5px solid {color};opacity:{0.6 if cerrada else 1};'
                'background:var(--background-fill-primary);border-radius:6px;padding:8px 10px;'
                'margin-bottom:8px;font-size:13px;line-height:1.5">'
                f'{etiqueta} <b>{f["id"]}</b> · {texto(f["proveedor"])}<br>'
                f'{texto(f["tipo"]).replace("_", " ")}<br>'
                + ('<span style="opacity:.7">✔ Cerrada</span>' if cerrada else
                   f'<span style="opacity:.7">Límite: {texto(f["fecha_limite"])}</span>'
                   + (' <b style="color:#e5484d">· VENCIDA</b>' if vencida else ""))
                + f'<div style="margin-top:4px;opacity:.85">{texto(f["resumen"])}</div></div>')
        columnas += (
            '<div style="flex:1;min-width:220px;background:var(--block-background-fill);'
            'border:1px solid var(--border-color-primary);border-radius:8px;padding:10px">'
            f'<div style="font-weight:600;margin-bottom:8px">{est} ({len(filas)})</div>'
            f'{tarjetas or "<div style=opacity:.5;font-size:13px>Sin incidencias</div>"}</div>')
    return f'<div style="display:flex;gap:12px;flex-wrap:wrap">{columnas}</div>'

def ui_seguimiento():
    ids = [f["id"] for f in REGISTRO]
    return (tabla_registro()[COLUMNAS_TABLA], kanban_html(), kpis(),
            gr.Dropdown(choices=ids, value=ids[-1] if ids else None))

def ui_cambiar_vista(vista):
    return gr.Dataframe(visible=vista == "Tabla"), gr.HTML(visible=vista == "Kanban")

def ui_analizar(texto):
    texto = (texto or "").strip()
    apagado = gr.Button(interactive=False)
    if len(texto) < 20:
        return "⚠️ Pega la incidencia (al menos una frase).", None, "", None, apagado, apagado, ""
    datos, avisos = analizar_incidencia(texto)
    if datos is None:
        return "❌ " + avisos[0], None, "", None, apagado, apagado, ""
    accion = proponer_accion(datos)
    encendido = gr.Button(interactive=True)
    return (ficha_markdown(datos, accion, avisos), datos, "",
            {"datos": datos, "accion": accion, "texto": texto}, encendido, encendido, "")

def ui_borrador(estado):
    return redactar_respuesta(estado["datos"], estado["texto"], estado["accion"])

def ui_registrar(estado):
    nuevo_id = registrar(estado["datos"], estado["accion"])
    msg = f"✅ Registrada como **{nuevo_id}** · responder antes del {estado['accion']['fecha_limite']}. Mírala en *Seguimiento*."
    return (msg, gr.Button(interactive=False)) + ui_seguimiento()   # se apaga para no registrarla dos veces

def ui_cambiar_estado(id_incidencia, estado):
    msg = cambiar_estado(id_incidencia, estado) if id_incidencia else "Elige una incidencia."
    return (msg,) + ui_seguimiento()

with gr.Blocks(title="Gestor de incidencias con proveedores") as app:
    gr.Markdown("# 📦 Gestor de incidencias con proveedores\n"
                "Pega la incidencia → la IA la clasifica → las reglas de Compras proponen acción y plazo → tú revisas y registras. "
                "*El modelo puede equivocarse: revisa siempre la ficha.*")
    estado_analisis = gr.State(None)

    with gr.Tab("📝 Nueva incidencia"):
        with gr.Row(equal_height=False):
            with gr.Column(scale=1):
                ejemplos = gr.Dropdown(choices=EJEMPLOS_APP, value=None, label="Cargar un ejemplo (opcional)")
                entrada = gr.Textbox(label="Incidencia", lines=8,
                                     placeholder="Pega aquí el correo del proveedor o la nota del almacén…")
                btn_analizar = gr.Button("Analizar", variant="primary")
            with gr.Column(scale=1):
                ficha = gr.Markdown("La ficha de la incidencia aparecerá aquí.")
                with gr.Accordion("Datos extraídos (JSON)", open=False):
                    salida_json = gr.JSON(show_label=False)
        with gr.Row():
            btn_borrador = gr.Button("✉️ Redactar respuesta al proveedor", interactive=False)
            btn_registrar = gr.Button("📥 Registrar incidencia", interactive=False)
        borrador = gr.Textbox(label="Borrador (edítalo aquí y cópialo con el botón de la esquina)", lines=8,
                              interactive=True, buttons=["copy"])
        msg_registro = gr.Markdown()

    with gr.Tab("📋 Seguimiento"):
        panel_kpis = gr.Markdown(kpis())
        vista = gr.Radio(["Kanban", "Tabla"], value="Kanban", label="Vista", show_label=False)
        kanban = gr.HTML(kanban_html())
        tabla = gr.Dataframe(value=tabla_registro()[COLUMNAS_TABLA], interactive=False, wrap=True, visible=False)
        with gr.Row():
            sel_id = gr.Dropdown(label="Incidencia", choices=[f["id"] for f in REGISTRO])
            sel_estado = gr.Dropdown(label="Nuevo estado", choices=ESTADOS, value="En curso")
            btn_estado = gr.Button("Actualizar estado")
        msg_estado = gr.Markdown()

    btn_analizar.click(ui_analizar, entrada,
                       [ficha, salida_json, borrador, estado_analisis, btn_borrador, btn_registrar, msg_registro])
    btn_borrador.click(ui_borrador, estado_analisis, borrador)
    btn_registrar.click(ui_registrar, estado_analisis, [msg_registro, btn_registrar, tabla, kanban, panel_kpis, sel_id])
    btn_estado.click(ui_cambiar_estado, [sel_id, sel_estado], [msg_estado, tabla, kanban, panel_kpis, sel_id])
    vista.change(ui_cambiar_vista, vista, [tabla, kanban])
    ejemplos.change(lambda texto: texto or "", ejemplos, entrada)   # elegir un ejemplo lo copia en la caja

app.launch(theme=gr.themes.Soft(), share=False, debug=False)   # share=True da un enlace público temporal

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

---
## Resumen Proyecto
- **Problema:** las incidencias con proveedores llegan desordenadas; se clasifican a ojo y el seguimiento se pierde.
- **Usuario:** comprador o analista de compras de una empresa industrial o de distribución.
- **Solución:** el LLM extrae y clasifica; las reglas deciden acción y plazo; el LLM redacta el borrador; la persona revisa; el registro hace el seguimiento.
- **Cómo lo evalué:** con las 20 incidencias etiquetadas a mano, Qwen3-4B acertó el 100 % del tipo y el 80 % de la gravedad, con 20/20 JSON válidos y 0 nº de pedido inventados (~9 s por incidencia, 0 €).
- **Límites:** datos pequeños y sintéticos; registro en CSV, no multiusuario; sin conexión al correo ni al ERP; el plazo no tiene en cuenta el importe; un texto que no es una incidencia con un proveedor puede colarse como tal.

